# NeoOLAF × RAGTree — FULL EventStoryLine → FinCausal

This notebook runs the two accepted configurations **sequentially and fully**:

1. **EventStoryLine** — 443 normalized records — frozen `v1.7`
2. **FinCausal** — 967 normalized records — frozen `unified-v1.3.1-selection-hotfix`

It is intentionally separate from the development/smoke controller.

### Scientific / budget guarantees

- The development manifest is **read-only** here; the full run does not alter the one-doc/smoke decisions.
- The notebook refuses to start unless the completed smoke state and frozen version for both datasets are present.
- Gold fields are stripped before Layer 0 (`entities`, `relations`, `pred_relations`, `ontology_links`).
- Gold is written/evaluated only **after native Layer 12 returns**.
- Each record has its own run directory.
- A persistent full-run progress file makes the experiment resumable.
- Successfully completed records are never paid for again unless you deliberately delete/change the progress state.
- A partially failed record is cleaned and retried on the next invocation.
- One isolated failure does not destroy the experiment; **3 consecutive failures stop the run** to avoid burning requests on a systematic problem.
- Dataset order is fixed: EventStoryLine first, then FinCausal.
- Exact UTC start/end windows are persisted for later OpenRouter usage/token accounting.

The OpenRouter API key is read only from `OPENROUTER_API_KEY` and is never printed or written to disk.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, time, traceback, shutil, re
from pprint import pprint

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def find_project_root():
    candidates = []
    env = os.environ.get("NEOOLAF_PROJECT_ROOT")
    if env:
        candidates.append(Path(env))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path(r"C:\Users\galencarmedeiro\NeoOLAF"))
    for p in candidates:
        if (p / "src" / "neoolaf").is_dir() and (p / "examples").is_dir():
            return p.resolve()
    raise FileNotFoundError(
        "NeoOLAF project root not found. Set NEOOLAF_PROJECT_ROOT."
    )

PROJECT_ROOT = find_project_root()
EXPERIMENT_ROOT = PROJECT_ROOT / "examples" / "RAGTreeDatasets"
TOOLS_DIR = EXPERIMENT_ROOT / "tools"

for p in [PROJECT_ROOT, PROJECT_ROOT / "src", TOOLS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ragtree_experiment_state_v1 as expstate
import ragtree_dataset_adapters_v1_8 as adapters
import eventstoryline_native_ablation_v1_7 as esl_v17

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Adapter:")
pprint(adapters.offline_self_test())


## Run controls

`RUN_PAID=True` is intentional: this notebook is the full benchmark run you requested.

The controller is serial at document level (`DOCUMENT_WORKERS=1`) so EventStoryLine finishes before FinCausal and progress accounting remains simple/reliable. Native layer-level parallelism remains enabled.


In [ ]:
RUN_PAID = True
RUN_MODE = "full"

# Fixed order: smaller normalized dataset first.
RUN_ORDER = ["eventstoryline", "fincausal"]

MODEL_NAME = "openai/gpt-oss-20b"
OPENROUTER_HOST = "https://openrouter.ai/api/v1"
REASONING_EFFORT = "minimal"
MAX_TOKENS = 8192
REQUEST_TIMEOUT = 180

DOCUMENT_WORKERS = 1
LAYER_WORKERS = 4
VERBOSE = True

# Resume / safety controls.
CHECKPOINT_EVERY = 10
MAX_CONSECUTIVE_FAILURES = 3
STOP_AFTER_N_PER_DATASET = None   # None = full dataset. Set an int only for a deliberate partial run.
OVERWRITE_COMPLETED = False       # Keep False for the real experiment.

EXPECTED_VERSIONS = {
    "eventstoryline": "v1.7",
    "fincausal": "unified-v1.3.1-selection-hotfix",
}

assert RUN_ORDER == ["eventstoryline", "fincausal"]
assert DOCUMENT_WORKERS == 1
assert MAX_CONSECUTIVE_FAILURES >= 1

print("RUN_PAID:", RUN_PAID)
print("RUN_MODE:", RUN_MODE)
print("RUN_ORDER:", RUN_ORDER)
print("MODEL:", MODEL_NAME)


## Zero-cost path, ontology, dataset and frozen-version preflight

This cell makes no model/API calls. It verifies the exact normalized inputs, the two ontology seeds, the two frozen configs, the completed smoke state, and pipeline-visible gold stripping.


In [ ]:
RAGTREE_ROOT = expstate.discover_ragtree_root(PROJECT_ROOT)
PREPROCESSED_DIR = expstate.discover_preprocessed_dir(RAGTREE_ROOT)
ONTOLOGY_ROOT = expstate.discover_ontology_dir(RAGTREE_ROOT)

RAW_ONTOLOGY_FILES = expstate.locate_ontology_files(ONTOLOGY_ROOT)
DATASET_FILES = expstate.locate_dataset_files(PREPROCESSED_DIR)

ONTOLOGY_FILES = {
    "eventstoryline": RAW_ONTOLOGY_FILES["eventstoryline"],
    "fincausal": RAW_ONTOLOGY_FILES["fincausal"],
}

CONFIGS = {
    "eventstoryline": {
        "profile": EXPERIMENT_ROOT / "configs/eventstoryline_profile_native_ablation_v1_7.json",
        "guidance": EXPERIMENT_ROOT / "configs/guidance_eventstoryline_native_ablation_v1_7.json",
        "task": EXPERIMENT_ROOT / "configs/eventstoryline_task_guidance_v1_7.json",
        "catalog": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_aliases.json",
        "version": "v1.7",
    },
    "fincausal": {
        "profile": EXPERIMENT_ROOT / "configs/fincausal_profile_unified_v1_3.json",
        "guidance": EXPERIMENT_ROOT / "configs/fincausal_guidance_unified_v1_3.json",
        "task": EXPERIMENT_ROOT / "configs/fincausal_task_guidance_unified_v1_3.json",
        "catalog": EXPERIMENT_ROOT / "ontology/fincausal_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/fincausal_relation_aliases.json",
        "version": "unified-v1.3.1-selection-hotfix",
    },
}

for k, cfg in CONFIGS.items():
    assert cfg["version"] == EXPECTED_VERSIONS[k], (k, cfg["version"])
    for name, p in cfg.items():
        if name != "version":
            assert Path(p).exists(), (k, name, p)

for k, p in ONTOLOGY_FILES.items():
    assert Path(p).exists(), (k, p)

dataset_rows = {
    k: expstate.read_jsonl(DATASET_FILES[k])
    for k in RUN_ORDER
}

# Exact full-run sizes currently expected from the normalized benchmark.
assert len(dataset_rows["eventstoryline"]) == 443, len(dataset_rows["eventstoryline"])
assert len(dataset_rows["fincausal"]) == 967, len(dataset_rows["fincausal"])

# Frozen development/smoke gate. READ-ONLY: this notebook never saves this manifest.
STATE_DIR = EXPERIMENT_ROOT / "state"
TEMPLATE_MANIFEST = STATE_DIR / "development_manifest_TEMPLATE_v1.json"
LIVE_MANIFEST = STATE_DIR / "development_manifest_v1.json"
dev_manifest = expstate.load_manifest(LIVE_MANIFEST, TEMPLATE_MANIFEST)

for k in RUN_ORDER:
    e = dev_manifest[k]
    assert e.get("one_doc_completed"), f"{k}: one-doc development gate is not complete"
    assert e.get("smoke5_already_run"), f"{k}: fixed smoke-5 is not complete"
    assert e.get("best_version") == EXPECTED_VERSIONS[k], (
        k, e.get("best_version"), EXPECTED_VERSIONS[k]
    )

# Ontology TBox check.
from neoolaf.ontology.loader import SeedOntologyLoader

def seed_counts(path):
    seed = SeedOntologyLoader().load(str(path))
    return {
        "classes": len(seed.classes_by_uri),
        "properties": len(seed.properties_by_uri),
    }

seed_info = {k: seed_counts(ONTOLOGY_FILES[k]) for k in RUN_ORDER}
assert seed_info["eventstoryline"]["classes"] > 0
assert seed_info["fincausal"]["classes"] >= 1000
assert seed_info["fincausal"]["properties"] >= 500

# Dataset audit + anti-leak check.
def audit_dataset(k, rows):
    rel_counts = {}
    entity_counts = []
    for r in rows:
        entity_counts.append(len(r.get("entities") or {}))
        for rel, pairs in (r.get("relations") or {}).items():
            rel_counts[rel] = rel_counts.get(rel, 0) + len(pairs or [])
    return {
        "dataset": k,
        "records": len(rows),
        "relations": rel_counts,
        "mean_gold_entities": (
            sum(entity_counts) / len(entity_counts) if entity_counts else 0.0
        ),
    }

audit = {k: audit_dataset(k, dataset_rows[k]) for k in RUN_ORDER}
assert {
    str(x).upper()
    for x in audit["eventstoryline"]["relations"]
    if str(x).lower() not in {"null", "none", ""}
} == {"PRECONDITION", "FALLING_ACTION"}
assert {
    str(x).upper()
    for x in audit["fincausal"]["relations"]
    if str(x).lower() not in {"null", "none", ""}
} == {"CAUSE"}

for k in RUN_ORDER:
    # Check every record, not just one sample.
    for r in dataset_rows[k]:
        clean = expstate.strip_gold(r)
        forbidden = {"entities", "relations", "pred_relations", "ontology_links"} & set(clean)
        assert not forbidden, (k, forbidden, r.get("document_id"))

print("RAGTREE_ROOT:", RAGTREE_ROOT)
print("PREPROCESSED_DIR:", PREPROCESSED_DIR)
print("\nFull input audit:")
pprint(audit)
print("\nSeed counts:")
pprint(seed_info)
print("\nFrozen smoke/version gate: OK")
print("Anti-leak check over all 1,410 records: OK")
print("No paid/API call has been made by this cell.")


## Persistent full-run state

The full experiment has its own state, independent of `development_manifest_v1.json`.

On restart:
- completed records are skipped;
- failed/incomplete records are retried;
- the controller refuses to silently change model, version, dataset size, or dataset order.

The state also stores exact UTC usage windows for later OpenRouter accounting.


In [ ]:
RUNS_ROOT = EXPERIMENT_ROOT / "runs" / "full_eventstoryline_then_fincausal_v1"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

PROGRESS_PATH = RUNS_ROOT / "full_progress.json"
LIVE_SUMMARY_PATH = RUNS_ROOT / "full_summary_live.json"
USAGE_WINDOW_PATH = RUNS_ROOT / "openrouter_usage_window.json"

def atomic_json(path, obj):
    expstate.atomic_write_json(Path(path), obj)

def load_json(path, default=None):
    p = Path(path)
    if not p.exists():
        return default
    return json.loads(p.read_text(encoding="utf-8"))

def fresh_progress():
    return {
        "schema_version": 1,
        "experiment": "NeoOLAF_FULL_EventStoryLine_then_FinCausal",
        "created_at_utc": utc_now(),
        "experiment_started_at_utc": None,
        "experiment_finished_at_utc": None,
        "model": MODEL_NAME,
        "host": OPENROUTER_HOST,
        "reasoning_effort": REASONING_EFFORT,
        "max_tokens": MAX_TOKENS,
        "dataset_order": list(RUN_ORDER),
        "datasets": {
            k: {
                "version": EXPECTED_VERSIONS[k],
                "total_records": len(dataset_rows[k]),
                "started_at_utc": None,
                "finished_at_utc": None,
                "completed_record_keys": [],
                "failures": {},
                "last_updated_at_utc": None,
            }
            for k in RUN_ORDER
        },
    }

progress = load_json(PROGRESS_PATH, None)
if progress is None:
    progress = fresh_progress()
    atomic_json(PROGRESS_PATH, progress)
else:
    # Refuse accidental experiment drift on resume.
    assert progress["model"] == MODEL_NAME
    assert progress["host"] == OPENROUTER_HOST
    assert progress["dataset_order"] == RUN_ORDER
    for k in RUN_ORDER:
        assert progress["datasets"][k]["version"] == EXPECTED_VERSIONS[k]
        assert progress["datasets"][k]["total_records"] == len(dataset_rows[k])

def write_usage_window():
    usage = {
        "experiment": progress["experiment"],
        "model": progress["model"],
        "host": progress["host"],
        "experiment_started_at_utc": progress.get("experiment_started_at_utc"),
        "experiment_finished_at_utc": progress.get("experiment_finished_at_utc"),
        "datasets": {
            k: {
                "version": progress["datasets"][k]["version"],
                "started_at_utc": progress["datasets"][k].get("started_at_utc"),
                "finished_at_utc": progress["datasets"][k].get("finished_at_utc"),
                "completed_records": len(progress["datasets"][k].get("completed_record_keys") or []),
                "total_records": progress["datasets"][k]["total_records"],
            }
            for k in RUN_ORDER
        },
        "note": (
            "Use these UTC windows + model=openai/gpt-oss-20b when inspecting "
            "OpenRouter activity/token usage. This file contains no API key."
        ),
    }
    atomic_json(USAGE_WINDOW_PATH, usage)
    return usage

write_usage_window()

print("RUNS_ROOT:", RUNS_ROOT)
print("PROGRESS_PATH:", PROGRESS_PATH)
for k in RUN_ORDER:
    ds = progress["datasets"][k]
    print(
        f"{k:15s}: completed={len(ds['completed_record_keys'])}/{ds['total_records']}, "
        f"recorded_failures={len(ds['failures'])}"
    )


## Native per-record runner

Each pending record gets a clean dedicated directory. A successful directory is never touched again by the controller.

For FinCausal, positive-gold records have an additional **post-Layer-12 evaluator-integrity guard**: if the controller knows the record has a scored CAUSE relation but the posthoc evaluator reports `gold=0`, the run is rejected as an evaluation bug rather than recorded as a zero score.


In [ ]:
def safe_dir_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))[:120]

def run_one_record(dataset_key, gold_record):
    cfg = CONFIGS[dataset_key]
    rkey = expstate.record_key(dataset_key, gold_record)

    # Controller-only contract. It is NOT passed to NeoOLAF.
    pre_gold_contract = expstate.gold_contract_summary(dataset_key, gold_record)

    run_dir = RUNS_ROOT / dataset_key / RUN_MODE / safe_dir_name(rkey)

    # Pending/failed records may have partial files. Clean only this record before retry.
    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    clean_record = expstate.strip_gold(gold_record)
    forbidden = {"entities", "relations", "pred_relations", "ontology_links"} & set(clean_record)
    assert not forbidden, (dataset_key, rkey, forbidden)

    input_path = run_dir / "pipeline_input_NO_GOLD.jsonl"
    expstate.write_jsonl(input_path, [clean_record])

    gold_path = run_dir / "POSTHOC_GOLD_AFTER_LAYER12.jsonl"
    assert not gold_path.exists()

    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    started = utc_now()
    t0 = time.perf_counter()

    if dataset_key == "eventstoryline":
        final_state = esl_v17.run_native_pipeline(
            project_root=PROJECT_ROOT,
            input_jsonl=input_path,
            ontology_path=ONTOLOGY_FILES[dataset_key],
            profile_path=cfg["profile"],
            guidance_path=cfg["guidance"],
            task_guidance_path=cfg["task"],
            relation_catalog_path=cfg["catalog"],
            relation_aliases_path=cfg["aliases"],
            run_dir=run_dir,
            model_name=MODEL_NAME,
            api_key=api_key,
            host=OPENROUTER_HOST,
            workers=LAYER_WORKERS,
            max_tokens=MAX_TOKENS,
            request_timeout=REQUEST_TIMEOUT,
            reasoning_effort=REASONING_EFFORT,
            verbose=VERBOSE,
            clean_run_dir=False,
        )

        # Gold appears only after Layer 12 returned.
        expstate.write_jsonl(
            gold_path,
            [{k: v for k, v in gold_record.items() if not k.startswith("__")}],
        )
        summary = esl_v17.analyze_run(
            run_dir=run_dir,
            gold_jsonl=gold_path,
            catalog_path=cfg["catalog"],
            aliases_path=cfg["aliases"],
        )
        relation_metrics = (
            summary.get("projected_relation_evaluation")
            or summary.get("strict_relation_evaluation")
            or {}
        )
        endpoint_metrics = (
            summary.get("relation_endpoint_evaluation")
            or summary.get("event_entity_evaluation")
            or {}
        )
        result = {
            "dataset": dataset_key,
            "version": cfg["version"],
            "record_key": rkey,
            "document_id": gold_record.get("document_id"),
            "title": gold_record.get("title"),
            "relation_metrics": relation_metrics,
            "endpoint_metrics": endpoint_metrics,
            "candidate_pool": (
                summary.get("candidate_pool_coverage")
                or summary.get("candidate_pool")
                or {}
            ),
            "run_dir": str(run_dir),
            "pre_run_gold_contract": pre_gold_contract,
        }

    elif dataset_key == "fincausal":
        final_state = adapters.run_native_pipeline_record(
            dataset_key=dataset_key,
            project_root=PROJECT_ROOT,
            input_jsonl=input_path,
            ontology_path=ONTOLOGY_FILES[dataset_key],
            profile_path=cfg["profile"],
            guidance_path=cfg["guidance"],
            task_guidance_path=cfg["task"],
            relation_catalog_path=cfg["catalog"],
            relation_aliases_path=cfg["aliases"],
            run_dir=run_dir,
            model_name=MODEL_NAME,
            api_key=api_key,
            host=OPENROUTER_HOST,
            workers=LAYER_WORKERS,
            max_tokens=MAX_TOKENS,
            request_timeout=REQUEST_TIMEOUT,
            reasoning_effort=REASONING_EFFORT,
            verbose=VERBOSE,
            clean_run_dir=False,
        )

        # Gold appears only after Layer 12 returned.
        expstate.write_jsonl(
            gold_path,
            [{k: v for k, v in gold_record.items() if not k.startswith("__")}],
        )
        result = adapters.evaluate_state(dataset_key, final_state, gold_record)
        result.update({
            "dataset": dataset_key,
            "version": cfg["version"],
            "record_key": rkey,
            "document_id": gold_record.get("document_id"),
            "title": gold_record.get("title"),
            "run_dir": str(run_dir),
            "pre_run_gold_contract": pre_gold_contract,
        })

        expected_gold = int(pre_gold_contract["gold_target_relation_count"])
        evaluated_gold = int((result.get("relation_metrics") or {}).get("gold", 0) or 0)
        if expected_gold > 0 and evaluated_gold == 0:
            raise RuntimeError(
                "FinCausal evaluation-integrity error AFTER Layer 12: "
                f"controller saw {expected_gold} gold CAUSE relation(s), evaluator saw 0. "
                "This is an evaluator/projection bug, not an extraction score."
            )
    else:
        raise ValueError(dataset_key)

    elapsed = time.perf_counter() - t0
    result["started_at_utc"] = started
    result["finished_at_utc"] = utc_now()
    result["elapsed_seconds"] = elapsed
    result["model"] = MODEL_NAME
    result["gold_visible_to_pipeline"] = False

    adapters.write_json(run_dir / "posthoc_evaluation.json", result)
    return result

print("Native full-run record runner defined. No API call has been made by this cell.")


## Result loading and aggregation

This is metric-schema tolerant: EventStoryLine uses names such as `true_positive` / `predicted`, while FinCausal uses `tp` / `pred`.

The full benchmark report includes:
- relation micro precision / recall / F1,
- macro document F1,
- macro F1 over positive-gold records,
- endpoint micro precision / recall / F1,
- completed / failed / pending counts,
- total wall-clock time accumulated from successful records.


In [ ]:
def _metric_count(m, *names):
    if not isinstance(m, dict):
        return 0
    for name in names:
        if name in m and m[name] is not None:
            return int(m[name] or 0)
    return 0

def normalize_counts(m):
    tp = _metric_count(m, "tp", "true_positive")
    fp = _metric_count(m, "fp", "false_positive")
    fn = _metric_count(m, "fn", "false_negative")
    pred = _metric_count(m, "pred", "predicted")
    gold = _metric_count(m, "gold", "gold_unique")

    if pred == 0 and (tp + fp) > 0:
        pred = tp + fp
    if gold == 0 and (tp + fn) > 0:
        gold = tp + fn

    return {"pred": pred, "gold": gold, "tp": tp, "fp": fp, "fn": fn}

def prf_from_counts(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f1

def result_path_for(dataset_key, rkey):
    return (
        RUNS_ROOT
        / dataset_key
        / RUN_MODE
        / safe_dir_name(rkey)
        / "posthoc_evaluation.json"
    )

def load_completed_results(dataset_key):
    completed = set(progress["datasets"][dataset_key]["completed_record_keys"])
    rows = []
    missing_files = []
    for r in dataset_rows[dataset_key]:
        rkey = expstate.record_key(dataset_key, r)
        if rkey not in completed:
            continue
        p = result_path_for(dataset_key, rkey)
        if not p.exists():
            missing_files.append((rkey, str(p)))
            continue
        rows.append(json.loads(p.read_text(encoding="utf-8")))
    if missing_files:
        raise RuntimeError(
            f"{dataset_key}: progress says records completed but result files are missing: "
            f"{missing_files[:3]}"
        )
    return rows

def aggregate_dataset(dataset_key):
    rows = load_completed_results(dataset_key)

    rel_counts = [normalize_counts(r.get("relation_metrics") or {}) for r in rows]
    ep_counts = [normalize_counts(r.get("endpoint_metrics") or {}) for r in rows]

    rel_tp = sum(x["tp"] for x in rel_counts)
    rel_fp = sum(x["fp"] for x in rel_counts)
    rel_fn = sum(x["fn"] for x in rel_counts)
    rel_pred = sum(x["pred"] for x in rel_counts)
    rel_gold = sum(x["gold"] for x in rel_counts)
    rel_p, rel_r, rel_f1 = prf_from_counts(rel_tp, rel_fp, rel_fn)

    ep_tp = sum(x["tp"] for x in ep_counts)
    ep_fp = sum(x["fp"] for x in ep_counts)
    ep_fn = sum(x["fn"] for x in ep_counts)
    ep_pred = sum(x["pred"] for x in ep_counts)
    ep_gold = sum(x["gold"] for x in ep_counts)
    ep_p, ep_r, ep_f1 = prf_from_counts(ep_tp, ep_fp, ep_fn)

    doc_f1s = [
        float((r.get("relation_metrics") or {}).get("f1", 0.0) or 0.0)
        for r in rows
    ]
    positive_doc_f1s = [
        float((r.get("relation_metrics") or {}).get("f1", 0.0) or 0.0)
        for r, c in zip(rows, rel_counts)
        if c["gold"] > 0
    ]

    ds_state = progress["datasets"][dataset_key]
    completed = len(ds_state["completed_record_keys"])
    total = ds_state["total_records"]
    failures = len(ds_state["failures"])
    elapsed = sum(float(r.get("elapsed_seconds", 0.0) or 0.0) for r in rows)

    return {
        "dataset": dataset_key,
        "version": EXPECTED_VERSIONS[dataset_key],
        "status": "COMPLETE" if completed == total else "INCOMPLETE",
        "completed_records": completed,
        "total_records": total,
        "pending_records": total - completed,
        "recorded_failures": failures,
        "relation": {
            "pred": rel_pred,
            "gold": rel_gold,
            "tp": rel_tp,
            "fp": rel_fp,
            "fn": rel_fn,
            "precision": rel_p,
            "recall": rel_r,
            "micro_f1": rel_f1,
            "macro_doc_f1": sum(doc_f1s) / len(doc_f1s) if doc_f1s else 0.0,
            "macro_positive_gold_doc_f1": (
                sum(positive_doc_f1s) / len(positive_doc_f1s)
                if positive_doc_f1s else 0.0
            ),
            "positive_gold_docs": len(positive_doc_f1s),
        },
        "endpoint": {
            "pred": ep_pred,
            "gold": ep_gold,
            "tp": ep_tp,
            "fp": ep_fp,
            "fn": ep_fn,
            "precision": ep_p,
            "recall": ep_r,
            "micro_f1": ep_f1,
        },
        "successful_record_wall_seconds_sum": elapsed,
        "dataset_started_at_utc": ds_state.get("started_at_utc"),
        "dataset_finished_at_utc": ds_state.get("finished_at_utc"),
    }

def save_live_summary():
    summary = {
        "experiment": progress["experiment"],
        "model": MODEL_NAME,
        "experiment_started_at_utc": progress.get("experiment_started_at_utc"),
        "experiment_finished_at_utc": progress.get("experiment_finished_at_utc"),
        "datasets": {k: aggregate_dataset(k) for k in RUN_ORDER},
    }
    atomic_json(LIVE_SUMMARY_PATH, summary)
    write_usage_window()
    return summary

print("Aggregation helpers ready. No API call has been made by this cell.")


## Execute the full experiment

This is the paid cell.

It processes all pending **EventStoryLine** records first. Only after EventStoryLine reaches 443/443 does it begin **FinCausal**.

Progress is saved after every record. A compact aggregate is checkpointed every 10 successful records. Re-running the notebook resumes from disk.


In [ ]:
if not RUN_PAID:
    print("RUN_PAID=False -> stopped before all model/API calls.")
else:
    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    if progress.get("experiment_started_at_utc") is None:
        progress["experiment_started_at_utc"] = utc_now()
        atomic_json(PROGRESS_PATH, progress)
        write_usage_window()

    global_abort = False

    for dataset_key in RUN_ORDER:
        if global_abort:
            break

        ds_state = progress["datasets"][dataset_key]
        total_rows = dataset_rows[dataset_key]
        completed = set(ds_state.get("completed_record_keys") or [])

        # Enforce strict dataset sequencing.
        if dataset_key == "fincausal":
            esl_state = progress["datasets"]["eventstoryline"]
            if len(esl_state.get("completed_record_keys") or []) != esl_state["total_records"]:
                raise RuntimeError(
                    "FinCausal is blocked until EventStoryLine is fully complete."
                )

        if len(completed) == len(total_rows) and not OVERWRITE_COMPLETED:
            print(f"\nSKIP {dataset_key}: already complete ({len(completed)}/{len(total_rows)}).")
            if ds_state.get("finished_at_utc") is None:
                ds_state["finished_at_utc"] = utc_now()
                atomic_json(PROGRESS_PATH, progress)
                write_usage_window()
            continue

        if ds_state.get("started_at_utc") is None:
            ds_state["started_at_utc"] = utc_now()
            ds_state["last_updated_at_utc"] = utc_now()
            atomic_json(PROGRESS_PATH, progress)
            write_usage_window()

        pending = []
        for r in total_rows:
            rkey = expstate.record_key(dataset_key, r)
            if OVERWRITE_COMPLETED or rkey not in completed:
                pending.append(r)

        if STOP_AFTER_N_PER_DATASET is not None:
            pending = pending[: int(STOP_AFTER_N_PER_DATASET)]

        print(
            f"\n=== FULL {dataset_key} ===\n"
            f"total={len(total_rows)} | already_completed={len(completed)} | "
            f"pending_now={len(pending)} | version={EXPECTED_VERSIONS[dataset_key]}"
        )

        consecutive_failures = 0

        for pending_idx, gold_record in enumerate(pending, 1):
            rkey = expstate.record_key(dataset_key, gold_record)
            absolute_done_before = len(ds_state.get("completed_record_keys") or [])

            print(
                f"\n[{dataset_key}] pending {pending_idx}/{len(pending)} | "
                f"overall completed {absolute_done_before}/{len(total_rows)}"
            )
            print("record_key:", rkey)
            print("document_id:", gold_record.get("document_id"))
            if gold_record.get("title") is not None:
                print("title:", gold_record.get("title"))

            try:
                result = run_one_record(dataset_key, gold_record)

                # Success is persisted immediately.
                completed_list = ds_state.setdefault("completed_record_keys", [])
                if rkey not in completed_list:
                    completed_list.append(rkey)

                ds_state.setdefault("failures", {}).pop(rkey, None)
                ds_state["last_updated_at_utc"] = utc_now()
                consecutive_failures = 0

                atomic_json(PROGRESS_PATH, progress)
                write_usage_window()

                print("relation:", result.get("relation_metrics"))
                print("endpoint:", result.get("endpoint_metrics"))
                print(f"elapsed_seconds: {result.get('elapsed_seconds', 0.0):.2f}")

                now_completed = len(ds_state["completed_record_keys"])
                if now_completed % CHECKPOINT_EVERY == 0 or now_completed == len(total_rows):
                    live = save_live_summary()
                    agg = live["datasets"][dataset_key]
                    print(
                        f"\nCHECKPOINT {dataset_key}: "
                        f"{agg['completed_records']}/{agg['total_records']} | "
                        f"relation micro-F1={agg['relation']['micro_f1']:.6f}"
                    )

            except Exception as exc:
                consecutive_failures += 1
                failure = {
                    "record_key": rkey,
                    "document_id": gold_record.get("document_id"),
                    "title": gold_record.get("title"),
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                    "traceback": traceback.format_exc(),
                    "failed_at_utc": utc_now(),
                }
                ds_state.setdefault("failures", {})[rkey] = failure
                ds_state["last_updated_at_utc"] = utc_now()

                atomic_json(PROGRESS_PATH, progress)
                write_usage_window()
                save_live_summary()

                print(
                    f"FAILED {dataset_key} {rkey}: "
                    f"{type(exc).__name__}: {exc}"
                )
                print(
                    f"Consecutive failures: "
                    f"{consecutive_failures}/{MAX_CONSECUTIVE_FAILURES}"
                )

                if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                    print(
                        "ABORTING after consecutive failures to avoid wasting API calls. "
                        "Fix the issue and rerun; completed records will be skipped."
                    )
                    global_abort = True
                    break

        # Dataset is complete only if every source record succeeded.
        completed = set(ds_state.get("completed_record_keys") or [])
        if len(completed) == len(total_rows):
            ds_state["finished_at_utc"] = utc_now()
            ds_state["last_updated_at_utc"] = utc_now()
            atomic_json(PROGRESS_PATH, progress)
            write_usage_window()
            save_live_summary()
            print(f"\n{dataset_key} COMPLETE: {len(completed)}/{len(total_rows)}")
        else:
            print(
                f"\n{dataset_key} INCOMPLETE: "
                f"{len(completed)}/{len(total_rows)} successful."
            )
            if global_abort:
                break

    # Overall completion.
    fully_complete = all(
        len(progress["datasets"][k].get("completed_record_keys") or [])
        == progress["datasets"][k]["total_records"]
        for k in RUN_ORDER
    )
    if fully_complete:
        progress["experiment_finished_at_utc"] = utc_now()
        atomic_json(PROGRESS_PATH, progress)
        print("\nFULL TWO-DATASET EXPERIMENT COMPLETE.")

    final_live = save_live_summary()
    print("\nLive/final summary:")
    pprint(final_live)
    print("\nProgress:", PROGRESS_PATH)
    print("Summary :", LIVE_SUMMARY_PATH)
    print("Usage window:", USAGE_WINDOW_PATH)


## Report results after the run — zero API calls

You can reopen the notebook later and execute cells through this point (without the paid execution cell) to load the persisted results.

The UTC window printed here is the one to use when checking the corresponding experiment activity/token usage in OpenRouter. Keep the model filter on `openai/gpt-oss-20b`.


In [ ]:
summary = save_live_summary()
usage_window = write_usage_window()

print("FULL BENCHMARK SUMMARY")
print("======================")
for k in RUN_ORDER:
    a = summary["datasets"][k]
    print(
        f"\n{k} [{a['version']}] "
        f"{a['completed_records']}/{a['total_records']} ({a['status']})"
    )
    print(
        " relation:",
        f"P={a['relation']['precision']:.6f}",
        f"R={a['relation']['recall']:.6f}",
        f"micro-F1={a['relation']['micro_f1']:.6f}",
        f"macro-doc-F1={a['relation']['macro_doc_f1']:.6f}",
        f"TP={a['relation']['tp']}",
        f"FP={a['relation']['fp']}",
        f"FN={a['relation']['fn']}",
    )
    print(
        " endpoint:",
        f"P={a['endpoint']['precision']:.6f}",
        f"R={a['endpoint']['recall']:.6f}",
        f"micro-F1={a['endpoint']['micro_f1']:.6f}",
    )
    print(" failed/pending:", a["recorded_failures"], "/", a["pending_records"])

print("\nOPENROUTER USAGE WINDOW")
print("=======================")
pprint(usage_window)

print("\nFiles to send back for analysis if convenient:")
print(" -", LIVE_SUMMARY_PATH)
print(" -", PROGRESS_PATH)
print(" -", USAGE_WINDOW_PATH)
print("\nYou can also simply send me this executed notebook.")
